In [34]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as scp

def pricing_asian_call(S, K, q, T, r, daily_vol, batch_size=1000, N=1000000, confidence_lvl=0.99, decimal=3):

    sigma = daily_vol*np.sqrt(252) #Annualized volatility

    steps = int(T*252)
    dt = T/steps

    # In order to limit RAM usage, we separate our realizations of the option in batches
    batch_size = 10000
    batch_num = N // batch_size #We round to the nearest whole number of batches needed to do N realizations

    Payoffs = []

    for _ in range(batch_num):

        Z=np.random.normal( size=(steps,batch_size) )
        increments = (r - q - sigma**2/2)*dt + sigma*np.sqrt(dt) * Z
        
        Paths = S * np.exp(np.vstack([np.zeros(batch_size), np.cumsum(increments, axis=0)]))
        Mean = np.mean(Paths, axis=0)

        for i in range(len(Mean)):
            Payoffs.append(max(Mean[i]-K,0))


    Call_price = np.mean(Payoffs)*np.exp(-r*T)
    sigma_payoff = np.std(Payoffs) * np.exp(-r*T)
    Error_margin = scp.stats.norm.ppf(1-(1-confidence_lvl)/2)* ( sigma_payoff / np.sqrt(N) )

    return(
    f"The option price is : {round(Call_price,decimal+1)} ± {round(Error_margin,decimal+1)}, at confidence level {confidence_lvl*100}% \n"
    f"In the following interval :  [{round(Call_price - Error_margin , decimal )} ; {round( Call_price + Error_margin, decimal)}]" )

In [ ]:
#Test with S=100 , K=120 , dividends=2% , T= 1year , risk free rate = 2% and daily vol = 1%
print(pricing_asian_call(100, 120, 0.02, 1, 0.02, 0.01, decimal=5, N=10000000))

The option price is : 0.09569 ± 0.000713, at confidence level 99.0% 
In the following interval :  [0.09498 ; 0.0964]
